# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** Entities in the Croissant dataset are **uniquely identified by their `@id`**. We will use these for all data access.

In [ ]:
# Discover available record sets, fields, and columns by listing their IDs and details

def print_record_sets(dataset):
    print('Record Sets:')
    for rset in dataset.record_sets():
        print(f"  @id: {rset['@id']}")
        print(f"    name: {rset.get('name','')}")
        print(f"    description: {rset.get('description','')}")
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"    Fields:")
        for field in fields:
            print(f"      - {field['@id']}")

print_record_sets(dataset)

# For demonstration, list the first few records for a record set if available
example_record_sets = [rset['@id'] for rset in dataset.record_sets()]
if example_record_sets:
    record_set_id = example_record_sets[0]    # pick the first available
    print(f"\nSample records for record set: {record_set_id}")
    for idx, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if idx >= 2:
            break
else:
    print('No record sets found in the dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All operations refer to components by their `@id`.


In [ ]:
# List all available record set @id's
record_sets = [rset['@id'] for rset in dataset.record_sets()]

dataframes = {}

# Load all records from each record set into a DataFrame
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_sets:
    selected_record_set_id = record_sets[0]  # Use the first record set for demonstration
    print(f"Fields in record set '@id' {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print('No record sets with data found.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalizing, or grouping. All references use field `@id`s.

In [ ]:
# For demonstration, we attempt EDA if numeric fields exist.
import numpy as np

if record_sets:
    df = dataframes[selected_record_set_id]
    # Identify numeric fields by pandas dtype
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        print('No numeric fields found in DataFrame for analysis.')
    else:
        # Pick the first numeric field
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field, e.g., first non-numeric field
        group_candidates = [c for c in df.columns if c not in numeric_fields]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(grouped_df.head())
        else:
            print('No categorical fields to group by.')
else:
    print('EDA not performed: No data available.')

## 5. Visualization
Visualize distributions or relationships between fields using the loaded DataFrame and standard Python data visualization tools.

In [ ]:
# Simple visualization example if numeric data available
import matplotlib.pyplot as plt
%matplotlib inline

if record_sets and numeric_fields:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.grid(True)
    plt.show()
else:
    print('No numeric field available for histogram.')

## 6. Conclusion
In this notebook, we demonstrated how to load, overview, and analyze a dataset described using the Croissant metadata standard with the `mlcroissant` library. Review the dataset documentation and schema for detailed field meanings, and adapt the EDA and visualization accordingly to your analysis goals.